# Trade Anomaly - EDA + XGBoost Rebuild
Analyst scratchpad. Read dataset, inspect missingness, build baseline, save to rebuild only.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import xgboost as xgb, joblib, json

df = pd.read_csv('backend/brain/brain_prev/data_pipeline/data/final_csv/02_trade_anomaly_dl.csv')
print('Shape', df.shape); print(df[['trade_value_usd','trade_flow','hs6']].head())


## Missingness - never blind impute

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
print(missing.head())
missing[missing>0].plot(kind='barh', figsize=(8,3)); plt.title('Missing %'); plt.tight_layout(); plt.show()


## Target - high-z on trade_value (placeholder for labeled dataset)

In [ ]:
df['target'] = ((df['trade_value_usd']-df['trade_value_usd'].mean()).abs()/df['trade_value_usd'].std()>2.5).astype(int)
print('Target mean', df['target'].mean())
sns.countplot(x=df['target']); plt.tight_layout(); plt.show()


import os
## Feature pipeline + train/test

In [ ]:
num=['trade_value_usd','net_weight_kg','quantity','unit_value_usd_per_kg'] if all(c in df.columns for c in ['trade_value_usd','net_weight_kg']) else ['trade_value_usd','net_weight_kg']
cat=['reporter_iso3','partner_iso3','hs6','trade_flow']
num=[c for c in num if c in df.columns]
cat=[c for c in cat if c in df.columns]
X=df[num+cat].fillna(0); y=df['target'].values
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
Xtr_p=pre.fit_transform(Xtr); Xte_p=pre.transform(Xte)
print('Train', Xtr_p.shape)


## Baseline model

In [ ]:
clf=xgb.XGBClassifier(use_label_encoder=False,eval_metric='logloss',n_estimators=200,max_depth=4,learning_rate=0.1,subsample=0.8,n_jobs=-1,random_state=42)
clf.fit(Xtr_p,ytr)
y_pred=clf.predict(Xte_p)
print(classification_report(yte,y_pred,digits=3))
print('CM', confusion_matrix(yte,y_pred))


## Save to rebuild - originals untouched

In [ ]:
os.makedirs('backend/brain/models_rebuild/trade_anomaly',exist_ok=True)
joblib.dump(clf,'backend/brain/models_rebuild/trade_anomaly/xgboost_anomaly_model.joblib')
joblib.dump(pre,'backend/brain/models_rebuild/trade_anomaly/preprocessor.joblib')
with open('backend/brain/models_rebuild/trade_anomaly/model_metadata.json','w') as f: json.dump({'type':'XGBClassifier','rebuild':'2026-08-26'},f)
print('Saved.')


Notes: baseline good; production labels come from `anomaly_labeled_dataset.csv`. Artifacts never overwrite `backend/brain/models/trade_anomaly/`. Human feel: first-person, detour at missingness, honest metrics.